# 06 - Chunk, embed and push documents to AI Search

## Required permissions

- **You:** **Search Service Contributor** + **Search Index Data Contributor** on Search, and **Cognitive Services OpenAI User** (or the workshop's **Foundry Owner**) on the parent Foundry resource. Python calls embeddings as you; project-only Foundry User is insufficient.
- **Facilitator:** provide the embedding deployment and quota. No Blob access, indexer, SharePoint permission or tenant administrator role is needed here.

**Attendee alias:** use your `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This lab creates only your `alice01-push-chunks` index; chunk/document IDs and vector configuration are scoped inside it. The model and Search service stay shared. Aliases prevent naming collisions, not enforce access isolation.

**Local text files -> Python chunking -> Azure OpenAI embeddings REST API -> Search documents REST API -> a separate push index**. Unlike notebook 04, this notebook creates no data source, skillset or indexer. The application owns extraction, chunk IDs, chunking, vectorization, retries and deletion. Every request and JSON payload is visible; no Search or OpenAI SDK wrapper is used.

Use the same original synthetic TXT documents, 1,000-character chunks, 200-character overlap, embedding deployment and index schema as notebook 04. This makes the approaches comparable, but a simple character slicer is not guaranteed to match Text Split's sentence-aware boundaries. This lab reads text files only; PDF, DOCX and image extraction are out of scope.

Your user needs Search Service Contributor and Search Index Data Contributor on Search, plus Cognitive Services OpenAI User on the **parent Foundry resource** for direct embedding calls. Foundry Owner on that resource also covers inference. Project-scoped Foundry User alone does not grant account-level OpenAI inference. Search's system-assigned identity needs embedding inference for query-time text vectorization (the workshop Bicep grants Cognitive Services User). Semantic ranker must be enabled; Bicep enables its free tier.

No Storage access is needed. Embedding requests send these synthetic texts to the deployed model and incur usage charges; semantic ranking has separate quota/billing. Use no sensitive data. Your `-push-chunks` index is separate from notebook 04's `-blob-chunks` index. Prefixes prevent naming collisions, not access by other workshop attendees.

## 1. Configure your lab

Run `az login --tenant <workshop-tenant-id>` and use the Azure IQ Workshop kernel. Run from the repository root or `notebooks` directory. This notebook runs independently of notebook 04 but reuses the same deployed Search service and `workshop-embedding` deployment (`text-embedding-3-small`).

Keep the model and 1,536 dimensions identical during ingestion and querying. If changing an existing vector field's dimensions, recreate only your own lab index; a schema update alone does not regenerate vectors.

In [1]:
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import quote

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT',
            'AZURE_AI_ACCOUNT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Run the deployment first.')
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
index_name = f'{prefix}-push-chunks'
api_version = '2025-09-01'
embedding_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
embedding_dimensions = 1536
semantic_configuration = 'chunk-semantic'
credential = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
embedding_token = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
print('Push index:', index_name)
print('Embedding deployment:', embedding_deployment)


Push index: memasanz-push-chunks
Embedding deployment: workshop-embedding


## 2. Create the separate push index

The schema mirrors notebook 04: chunk text, filename, provenance and a vector field. Here **Python**, not index projections, supplies `chunk_id` and `parent_id`. HNSW uses cosine similarity. The keyless vectorizer lets Search embed query text using its own managed identity. Semantic ranking prioritizes `title` and `chunk`.

In [2]:
index = {
    'name': index_name,
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'title', 'type': 'Edm.String', 'searchable': True},
        {'name': 'text_vector', 'type': 'Collection(Edm.Single)',
         'searchable': True, 'retrievable': True, 'dimensions': embedding_dimensions,
         'vectorSearchProfile': 'chunk-vector-profile'},
        {'name': 'source', 'type': 'Edm.ComplexType', 'fields': [
            {'name': 'file_name', 'type': 'Edm.String', 'searchable': True},
            {'name': 'url', 'type': 'Edm.String'},
        ]},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'chunk-hnsw', 'kind': 'hnsw',
                        'hnswParameters': {'metric': 'cosine'}}],
        'profiles': [{'name': 'chunk-vector-profile', 'algorithm': 'chunk-hnsw',
                      'vectorizer': 'chunk-vectorizer'}],
        'vectorizers': [{
            'name': 'chunk-vectorizer', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
                'modelName': embedding_model,
            },
        }],
    },
    'semantic': {
        'defaultConfiguration': semantic_configuration,
        'configurations': [{
            'name': semantic_configuration,
            'prioritizedFields': {
                'titleField': {'fieldName': 'title'},
                'prioritizedContentFields': [{'fieldName': 'chunk'}],
            },
        }],
    },
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=index, timeout=(10, 90),
)
response.raise_for_status()
print('Index ready:', index_name)


Index ready: memasanz-push-chunks


## 3. Read and chunk the source files in Python

Use a visible character-slicing loop: at most 1,000 characters, with 200 characters of overlap. These are characters, not tokens or physical pages. The original filenames contain only Search-key-safe letters and hyphens; their stems and chunk numbers give deterministic keys for repeat uploads. `source.url` is a repository-relative path, not a Blob URL or a promise of hosted access.

Rerunning with unchanged inputs replaces documents at the same keys. **Shortening/removing files or changing chunking can leave old keys behind.** Before those experiments, use the opt-in cleanup to delete only this push index, then rerun from section 2. A production application must track and delete obsolete chunks explicitly.

In [3]:
sample_files = sorted((root / 'data' / 'search').glob('*.txt'))
if len(sample_files) != 2:
    raise RuntimeError('Expected the two workshop sample documents under data/search.')
chunk_size = 1000
chunk_overlap = 200
if not 0 <= chunk_overlap < chunk_size:
    raise ValueError('Chunk overlap must be non-negative and smaller than chunk size.')
documents = []
for path in sample_files:
    text = path.read_text(encoding='utf-8')
    if not text.strip():
        raise ValueError(f'Empty source document: {path.name}')
    for chunk_number, start in enumerate(range(0, len(text), chunk_size - chunk_overlap)):
        end = min(start + chunk_size, len(text))
        documents.append({
            '@search.action': 'upload',
            'chunk_id': f'{path.stem}-{chunk_number}', 'parent_id': path.stem,
            'title': path.name, 'chunk': text[start:end],
            'source': {'file_name': path.name, 'url': path.relative_to(root).as_posix()},
        })
        if end == len(text):
            break
print('Files:', len(sample_files), '| chunks:', len(documents))
print(json.dumps(documents[0], indent=2))


Files: 2 | chunks: 6
{
  "@search.action": "upload",
  "chunk_id": "onboarding-guide-0",
  "parent_id": "onboarding-guide",
  "title": "onboarding-guide.txt",
  "chunk": "Northwind Workshop - Synthetic Onboarding Guide\n\nThis fictional document is provided for an Azure AI Search workshop. It is not guidance for any real employer and contains no personal or confidential information.\n\nBefore the first workshop session\nParticipants receive a schedule, a link to the learning material and a list of local prerequisites. Install a supported Python interpreter, Visual Studio Code, the Jupyter extension and Azure CLI. Sign in to the workshop tenant with an approved account. The facilitator grants access to the shared learning resources. Participants should not request subscription-wide administrator privileges to complete a notebook.\n\nWorkspace setup\nCreate a local virtual environment and install the workshop requirements. Select that environment as the Jupyter kernel. Read the environme

## 4. Generate embeddings through the REST API

Here the **signed-in user**, not Search, calls Azure OpenAI. Requests contain small batches of up to 16 chunks. The deployment name belongs in the URL; `modelName` in the index vectorizer is the underlying model name, not necessarily the deployment name. Each embedding is matched to its input using the response's `index`, then checked against the vector field's dimensions.

A 403 usually means missing account-level inference access; a 429 can indicate embedding quota. Errors are surfaced rather than silently skipped. No embeddings are regenerated inside the following push cell.

In [4]:
for start in range(0, len(documents), 16):
    batch = documents[start:start + 16]
    response = requests.post(
        f'{embedding_endpoint}/openai/deployments/{quote(embedding_deployment, safe="")}/embeddings?api-version=2024-10-21',
        headers={'Authorization': f'Bearer {embedding_token()}'},
        json={'input': [item['chunk'] for item in batch], 'dimensions': embedding_dimensions},
        timeout=(10, 90),
    )
    response.raise_for_status()
    embeddings = response.json()['data']
    if sorted(item['index'] for item in embeddings) != list(range(len(batch))):
        raise RuntimeError('Embedding response did not contain exactly one vector per input.')
    for item in embeddings:
        vector = item['embedding']
        if len(vector) != embedding_dimensions:
            raise RuntimeError('Embedding dimensions do not match the Search vector field.')
        batch[item['index']]['text_vector'] = vector
print('Embedded chunks:', len(documents), '| dimensions:', embedding_dimensions)


Embedded chunks: 6 | dimensions: 1536


## 5. Push enriched chunks to Search

`POST /indexes/{name}/docs/index` receives the text, vectors, keys and provenance with `@search.action='upload'`. Small batches of 100 fit this lab's 1,536-dimension vectors comfortably within Search's 1,000-document/16-MB limits. For larger real-world inputs, also enforce the encoded request size.

A 207 response can mean some documents failed. Check every returned document's `status`, not just the HTTP status. A failed batch may already have written some documents: correct the error and rerun the push with the same keys; do not generate new keys to hide failures. No automated retry loop is added.

In [5]:
if not documents or any(len(item.get('text_vector') or []) != embedding_dimensions for item in documents):
    raise RuntimeError('Chunk and embed every document before pushing.')
for start in range(0, len(documents), 100):
    batch = documents[start:start + 100]
    response = requests.post(
        f'{search_endpoint}/indexes/{index_name}/docs/index?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'},
        json={'value': batch}, timeout=(10, 90),
    )
    response.raise_for_status()
    indexing_results = response.json()['value']
    if (sorted(item['key'] for item in indexing_results) != sorted(item['chunk_id'] for item in batch)
            or any(item.get('status') is not True for item in indexing_results)):
        raise RuntimeError(f'Push was incomplete or some documents failed: {indexing_results}')
    print('Uploaded chunks:', len(indexing_results))


Uploaded chunks: 6


## 6. Verify ingestion

Indexing is asynchronous. Poll each uploaded key until its stored text and vector match the document we sent, rather than accepting a successful batch response as proof that search is ready. This also avoids mistaking an older document at the same key for the new content. Print only IDs and vector lengths, not full vectors.

In [6]:
for attempt in range(30):
    pending = []
    for document in documents:
        key = quote(document['chunk_id'], safe='')
        response = requests.get(
            f'{search_endpoint}/indexes/{index_name}/docs/{key}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code == 404:
            pending.append(document['chunk_id'])
            continue
        response.raise_for_status()
        stored = response.json()
        vector = stored.get('text_vector') or []
        if (stored.get('chunk') != document['chunk'] or len(vector) != embedding_dimensions
                or any(abs(actual - expected) > 1e-6
                       for actual, expected in zip(vector, document['text_vector']))):
            pending.append(document['chunk_id'])
    if not pending:
        break
    if attempt < 29:
        time.sleep(2)
else:
    raise TimeoutError(f'Pushed documents are not yet visible with the expected content: {pending}')
for document in documents:
    print(document['chunk_id'], '| vector dimensions:', embedding_dimensions)


onboarding-guide-0 | vector dimensions: 1536
onboarding-guide-1 | vector dimensions: 1536
onboarding-guide-2 | vector dimensions: 1536
travel-policy-0 | vector dimensions: 1536
travel-policy-1 | vector dimensions: 1536
travel-policy-2 | vector dimensions: 1536


## 7. Compare keyword, vector, hybrid and semantic-ranked search

Use the same question and `top=3` as notebook 04. Keyword uses BM25, vector uses embedding similarity, and hybrid fuses their ranks with RRF. Semantic ranking reranks candidates and returns extractive captions; it is not answer generation. Compare sources and ordering, not raw scores between modes. The small dataset may yield identical rankings.

During ingestion Python generated vectors. Below, `kind='text'` delegates **query-time** embeddings to the Search vectorizer and Search's identity. This does not turn push ingestion into an indexer pipeline.

### Keyword search

In [7]:
question = 'What receipts do I need to claim travel expenses, and when must I submit them?'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': question, 'searchFields': 'chunk',
          'select': 'chunk_id,title,chunk,source', 'top': 3},
    timeout=(10, 90),
)
response.raise_for_status()
print(json.dumps(response.json(), indent=2))


{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes('megan03-push-chunks')/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 2.0811255,
      "chunk_id": "travel-policy-0",
      "chunk": "Northwind Workshop - Synthetic Travel Policy\n\nThis document is fictional workshop material. It contains no real employee records or corporate policy. Use it to explore indexing, chunk boundaries, provenance and keyword retrieval.\n\nTravel planning\nEmployees should discuss the purpose, expected outcomes and estimated cost of a trip with their manager before booking. The fictional Northwind Workshop organization prefers remote meetings when they achieve the same result. A trip request includes destination, travel dates, transport options and a short explanation of the customer or training activity. Manager approval should be recorded before any nonrefundable booking.\n\nTravel reimbursement and receipts\nSubmit travel reimbursement requests 

### Vector-only search

No `search` property: there are no keyword candidates in this request.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
response.raise_for_status()
print(json.dumps(response.json(), indent=2))


### Hybrid search (keyword + vector)

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
response.raise_for_status()
print(json.dumps(response.json(), indent=2))


### Keyword search + semantic ranking

Inspect `@search.rerankerScore` and `@search.captions`. Do not present a partial response as successful reranking.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk', 'queryType': 'semantic',
        'semanticConfiguration': semantic_configuration, 'captions': 'extractive',
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
response.raise_for_status()
if response.status_code != 200 or response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
print(json.dumps(response.json(), indent=2))


### Hybrid search + semantic ranking

`k=50` supplies up to 50 vector candidates before reranking; the sample corpus contains fewer.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'queryType': 'semantic', 'semanticConfiguration': semantic_configuration,
        'captions': 'extractive', 'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
response.raise_for_status()
if response.status_code != 200 or response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
print(json.dumps(response.json(), indent=2))


## 8. Compare ingestion approaches

| Responsibility | 04: Blob indexer | 05: Push API |
|---|---|---|
| Read/extract text | Search reads Blob files | Python reads local TXT files |
| Chunk documents | Text Split skill | Visible Python slicing loop |
| Generate document embeddings | Embedding skill, Search identity | Embeddings REST call, user identity |
| Assign keys and write chunks | Index projections + indexer | Python + documents REST API |
| Detect changes and remove old chunks | Indexer change tracking; configure deletion detection | Application tracks changes/deletions |
| Embed query text | Search vectorizer | Search vectorizer |
| Retrieve/rerank | Keyword, vector, hybrid, semantic | Same search features, separate index |

Compare chunk counts and boundaries, sources and query results in the two indexes. Explain why adding a vectorizer alone does not generate document vectors, and why semantic ranking is distinct from vector retrieval. Experiment with smaller chunks only after deleting/recreating this push index to avoid stale keys.

## 9. Optional cleanup

Delete only your named `-push-chunks` index. This never deletes notebook 04's index, any blobs, the shared Search service or model deployment. After cleanup, rerun from section 2 to rebuild.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    response = requests.delete(
        f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    if response.status_code != 404:
        response.raise_for_status()
    print('Deleted or already absent:', index_name)
else:
    print('Cleanup disabled.')


## References

- [REST index/skillset style reference](https://github.com/memasanz/mm-ai-search/blob/main/000_CreateIndex.ipynb)
- [Generate embeddings](https://learn.microsoft.com/azure/search/vector-search-how-to-generate-embeddings)
- [Load an index with the push APIs](https://learn.microsoft.com/azure/search/search-how-to-load-search-index)
- [Azure OpenAI vectorizer](https://learn.microsoft.com/azure/search/vector-search-vectorizer-azure-open-ai)
- [Hybrid search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Semantic ranking](https://learn.microsoft.com/azure/search/semantic-search-overview)

Next: SharePoint indexed ingestion with document-level permissions in notebook 07.